# ENIGMA - SpatialStressCNN 2D Training
**Train :** /content/drive/MyDrive/Data/datasets/train
**Val   :** /content/drive/MyDrive/Data/datasets/val
**Out   :** /content/drive/MyDrive/Data/model/

> Runtime -> Change runtime type -> **T4 GPU** before running!

## Step 1: Install Dependencies

In [ ]:
!pip install rasterio -q
import torch, os, time, joblib
import numpy as np
print("Torch: " + torch.__version__)
print("CUDA : " + str(torch.cuda.is_available()))

## Step 2: Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

TRAIN_DIR = "/content/drive/MyDrive/Data/datasets/train"
VAL_DIR   = "/content/drive/MyDrive/Data/datasets/val"
MODEL_DIR = "/content/drive/MyDrive/Data/model/"
os.makedirs(MODEL_DIR, exist_ok=True)

for name, path in [("Train", TRAIN_DIR), ("Val", VAL_DIR), ("Model out", MODEL_DIR)]:
    status = "OK" if os.path.exists(path) else "NOT FOUND"
    print(name.ljust(12) + ": " + status + "  ->  " + path)

print("")
for split_name, split_path in [("TRAIN", TRAIN_DIR), ("VAL", VAL_DIR)]:
    if os.path.exists(split_path):
        print(split_name + ":")
        for cls in sorted(os.listdir(split_path)):
            cls_path = os.path.join(split_path, cls)
            if os.path.isdir(cls_path):
                n = len([f for f in os.listdir(cls_path) if f.endswith(".tif")])
                print("  " + cls.ljust(15) + str(n) + " .tif files")

## Step 3: Configuration

In [ ]:
PATCH_SIZE    = 16
STRIDE        = 8
EPOCHS        = 25
BATCH_SIZE    = 256
LEARNING_RATE = 3e-4
MIN_NONZERO   = 0.3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device : " + str(device))
if torch.cuda.is_available():
    print("GPU    : " + torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("VRAM   : " + str(round(vram, 1)) + " GB")

## Step 4: Model Architecture

In [ ]:
import torch.nn as nn

class SpatialStressCNN(nn.Module):
    def __init__(self, num_bands, num_classes=3):
        super(SpatialStressCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(num_bands, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        flat_size = 128 * (PATCH_SIZE // 4) * (PATCH_SIZE // 4)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

print("Architecture defined")

## Step 5: TIFF Patch Dataset

In [ ]:
import rasterio
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

class TIFFPatchDataset(Dataset):
    def __init__(self, root_dir, patch_size=16, stride=8, min_nonzero=0.3,
                 num_bands_override=None, max_val_override=None):
        self.patches    = []
        self.labels     = []
        self.patch_size = patch_size
        classes         = sorted([d for d in os.listdir(root_dir)
                                  if os.path.isdir(os.path.join(root_dir, d))])
        self.le         = LabelEncoder().fit(classes)

        # Pass 1: detect max bands
        if num_bands_override is None:
            print("Pass 1/2 - scanning band counts...")
            max_bands = 0
            for cls in classes:
                for img in os.listdir(os.path.join(root_dir, cls)):
                    if img.endswith(".tif"):
                        try:
                            with rasterio.open(os.path.join(root_dir, cls, img)) as src:
                                max_bands = max(max_bands, src.count)
                        except:
                            continue
            self.num_bands = max_bands
        else:
            self.num_bands = num_bands_override

        print("Bands: " + str(self.num_bands) + "  Classes: " + str(classes))

        # Pass 2: extract patches
        print("Pass 2/2 - extracting patches...")
        skipped = 0
        for cls in classes:
            cls_dir   = os.path.join(root_dir, cls)
            label_idx = self.le.transform([cls])[0]
            imgs      = [f for f in os.listdir(cls_dir) if f.endswith(".tif")]
            print("  [" + cls + "] " + str(len(imgs)) + " images")
            for img_name in imgs:
                try:
                    with rasterio.open(os.path.join(cls_dir, img_name)) as src:
                        data = src.read().astype(np.float32)
                        c, h, w = data.shape
                        # Clean raw data
                        data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
                        # Spatial padding
                        if h < patch_size or w < patch_size:
                            ph = max(0, patch_size - h)
                            pw = max(0, patch_size - w)
                            data = np.pad(data, ((0,0),(0,ph),(0,pw)), mode="reflect")
                            _, h, w = data.shape
                        # Channel padding/trim
                        if c < self.num_bands:
                            data = np.pad(data, ((0, self.num_bands - c),(0,0),(0,0)), mode="constant")
                        elif c > self.num_bands:
                            data = data[:self.num_bands]
                        # Sliding window
                        for r in range(0, h - patch_size + 1, stride):
                            for col in range(0, w - patch_size + 1, stride):
                                patch = data[:, r:r+patch_size, col:col+patch_size]
                                if np.count_nonzero(patch) > patch.size * min_nonzero:
                                    self.patches.append(patch)
                                    self.labels.append(label_idx)
                                else:
                                    skipped += 1
                except Exception as e:
                    print("    SKIP " + img_name + " : " + str(e))

        print("Skipped low-signal patches: " + str(skipped))

        if len(self.patches) == 0:
            raise RuntimeError("No patches extracted! Check your data path.")

        self.patches = np.array(self.patches, dtype=np.float32)
        self.labels  = np.array(self.labels,  dtype=np.int64)

        # Normalise to [0, 1]
        if max_val_override is None:
            nonzero_vals = self.patches[self.patches > 0]
            self.max_val = float(np.percentile(nonzero_vals, 99)) if len(nonzero_vals) > 0 else 1.0
        else:
            self.max_val = max_val_override

        self.patches = np.clip(self.patches / max(self.max_val, 1e-6), 0.0, 1.0)
        self.patches = np.nan_to_num(self.patches, nan=0.0, posinf=1.0, neginf=0.0)

        print("Total patches : " + str(len(self.patches)))
        print("max_val       : " + str(round(self.max_val, 2)))
        unique, counts = np.unique(self.labels, return_counts=True)
        for u, cnt in zip(unique, counts):
            print("  " + str(self.le.classes_[u]).ljust(12) + ": " + str(cnt))

    def __len__(self):  return len(self.labels)
    def __getitem__(self, idx):
        return torch.tensor(self.patches[idx]), torch.tensor(self.labels[idx])

print("Dataset class ready")

## Step 6: Load Data, Train & Save

> All in one cell — no cross-cell variable issues.

In [ ]:
import torch.optim as optim

# ── LOAD TRAIN ────────────────────────────────────────────────
print("=" * 55)
print("Loading TRAIN dataset...")
print("=" * 55)
train_dataset = TIFFPatchDataset(
    TRAIN_DIR,
    patch_size  = PATCH_SIZE,
    stride      = STRIDE,
    min_nonzero = MIN_NONZERO,
)

# ── LOAD VAL (inherits num_bands + max_val from train) ────────
print("")
print("=" * 55)
print("Loading VAL dataset...")
print("=" * 55)
val_dataset = TIFFPatchDataset(
    VAL_DIR,
    patch_size         = PATCH_SIZE,
    stride             = PATCH_SIZE,
    min_nonzero        = MIN_NONZERO,
    num_bands_override = train_dataset.num_bands,
    max_val_override   = train_dataset.max_val,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print("")
print("Train patches : " + str(len(train_dataset)))
print("Val   patches : " + str(len(val_dataset)))
print("Num bands     : " + str(train_dataset.num_bands))
print("Classes       : " + str(list(train_dataset.le.classes_)))

# ── BUILD MODEL ───────────────────────────────────────────────
num_classes = len(train_dataset.le.classes_)
model       = SpatialStressCNN(train_dataset.num_bands, num_classes).to(device)
criterion   = nn.CrossEntropyLoss()
optimizer   = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("")
print("Trainable params : " + str(total_params))
print("-" * 62)

# ── TRAINING LOOP ─────────────────────────────────────────────
best_acc = 0.0
history  = {"loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    t0 = time.time()

    model.train()
    total_loss  = 0.0
    nan_batches = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out  = model(xb)
        loss = criterion(out, yb)
        if torch.isnan(loss):
            nan_batches += 1
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / max(len(train_loader) - nan_batches, 1)
    history["loss"].append(avg_loss)

    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            _, pred = torch.max(model(xb.to(device)), 1)
            correct += (pred == yb.to(device)).sum().item()
    acc = 100.0 * correct / len(val_dataset)
    history["val_acc"].append(acc)
    scheduler.step()

    flag = " << BEST" if acc > best_acc else ""
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, "stress_cnn_best.pt"))

    nan_warn = "" if nan_batches == 0 else "  [" + str(nan_batches) + " NaN batches]"
    elapsed  = round(time.time() - t0, 1)
    print("Epoch " + str(epoch+1).rjust(2) + "/" + str(EPOCHS) +
          "  Loss: " + str(round(avg_loss, 4)) +
          "  Val: "  + str(round(acc, 2)) + "%" +
          "  " + str(elapsed) + "s" + flag + nan_warn)

print("-" * 62)
print("Best val accuracy: " + str(round(best_acc, 2)) + "%")

# ── SAVE ALL OUTPUTS ──────────────────────────────────────────
torch.save(model.state_dict(),     os.path.join(MODEL_DIR, "stress_cnn.pt"))
joblib.dump(train_dataset.le,       os.path.join(MODEL_DIR, "cnn_label_encoder.pkl"))
joblib.dump(train_dataset.max_val,  os.path.join(MODEL_DIR, "cnn_scaler.pkl"))

print("")
print("Saved to: " + MODEL_DIR)
for fname in ["stress_cnn.pt", "stress_cnn_best.pt", "cnn_label_encoder.pkl", "cnn_scaler.pkl"]:
    p = os.path.join(MODEL_DIR, fname)
    if os.path.exists(p):
        print("  OK   " + fname.ljust(35) + str(round(os.path.getsize(p)/1e3, 1)) + " KB")
    else:
        print("  MISS " + fname)

## Step 7: Training Curve

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("SpatialStressCNN 2D - Training Summary", fontsize=13)

epochs_x = range(1, len(history["loss"]) + 1)
ax1.plot(epochs_x, history["loss"], marker="o", color="tomato")
ax1.set_title("Training Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_x, history["val_acc"], marker="o", color="steelblue")
ax2.axhline(best_acc, color="green", linestyle="--", alpha=0.7,
            label="Best: " + str(round(best_acc, 2)) + "%")
ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(MODEL_DIR, "training_curve.png")
plt.savefig(plot_path, dpi=120, bbox_inches="tight")
plt.show()
print("Saved: " + plot_path)

## Done!

Files in /content/drive/MyDrive/Data/model/

| File | Purpose |
|------|---------|
| stress_cnn.pt | Final weights |
| stress_cnn_best.pt | Best checkpoint |
| cnn_label_encoder.pkl | Label encoder for inference |
| cnn_scaler.pkl | max_val scalar for inference |
| training_curve.png | Loss and accuracy plot |